# TE - Visualização de Dados

## Pré-processamento dos LEM 2021 a 2025

Este notebook consolida as planilhas LEM da Fundação Pão dos Pobres em um único dataset padronizado, organizado em formato longo e pronto para análise exploratória e construção do dashboard.

**Entrada esperada:** arquivos `.xlsx` dentro da pasta `LEM`, mantendo apenas uma versão por ano.

**Saída gerada:** `dados_tratados/lem_consolidado_2021_2025.csv`.

## 1. Importação das bibliotecas

In [3]:
import pandas as pd
import numpy as np

from pathlib import Path
import re

## 2. Leitura dos arquivos Excel

O código abaixo procura a pasta `LEM` a partir do diretório atual do notebook. Caso ela não seja encontrada, ele procura arquivos `.xlsx` no próprio projeto.

In [4]:
#CARREGANDO OS DADOS

arquivos_excel = list(Path.cwd().rglob("*.xlsx"))

print(f"Foram encontrados {len(arquivos_excel)} arquivos Excel")

pasta_dados = arquivos_excel[0].parent

planilhas = {}

for arquivo in arquivos_excel:
    planilhas[arquivo.name] = pd.read_excel(arquivo)

print("Arquivos carregados com sucesso!")

Foram encontrados 5 arquivos Excel
Arquivos carregados com sucesso!


## 3. Inspeção resumida das abas

Esta etapa ajuda a confirmar se os arquivos estão sendo lidos corretamente.

In [5]:
resumo_abas = []

for arquivo in arquivos_excel:
    excel = pd.ExcelFile(arquivo)

    for aba in excel.sheet_names:
        df_aba = pd.read_excel(arquivo, sheet_name=aba, header=None)
        resumo_abas.append({
            "arquivo": arquivo.name,
            "aba": aba,
            "linhas": df_aba.shape[0],
            "colunas": df_aba.shape[1],
            "vazia": df_aba.dropna(how="all").empty
        })

resumo_abas_df = pd.DataFrame(resumo_abas)
resumo_abas_df

,arquivo,aba,linhas,colunas,vazia
0,LEM 2021.xlsx,Planilha1,44,21,False
1,LEM 2024.xlsx,Planilha1,43,21,False
2,LEM 2024.xlsx,Plan1,0,0,True
3,LEM 2025.xlsx,Planilha1,43,21,False
4,LEM 2025.xlsx,Plan1,0,0,True
5,LEM 2022.xlsx,Planilha1,43,21,False
6,LEM 2023.xlsx,Planilha1,43,21,False
7,LEM 2023.xlsx,Plan1,0,0,True


## 4. Padronização dos Meses

In [6]:
# Meses utilizados nas planilhas LEM
meses = ["JAN", "FEV", "MAR", "ABR", "MAI", "JUN", "JUL", "AGO", "SET", "OUT", "NOV", "DEZ"]

mapa_meses = {
    "JAN": 1,
    "FEV": 2,
    "MAR": 3,
    "ABR": 4,
    "MAI": 5,
    "JUN": 6,
    "JUL": 7,
    "AGO": 8,
    "SET": 9,
    "OUT": 10,
    "NOV": 11,
    "DEZ": 12,
}

categorias_lem = [
    "DESDOBRAMENTOS TÉCNICOS",
    "PROFISSIONALIZAÇÃO",
    "SAÚDE",
    "EDUCAÇÃO",
]

# Subcategorias encontradas principalmente dentro da categoria EDUCAÇÃO
subcategorias_educacao = [
    "Ensino infantil",
    "Ensino regular",
    "Ensino EJA",
    "SCFV",
]

# Subcategorias/linhas que podem aparecer dentro de outras seções
subcategorias_gerais = subcategorias_educacao

In [7]:
def limpar_texto(texto):
    """Remove espaços extras e padroniza valores textuais."""
    if pd.isna(texto):
        return None

    texto = str(texto).strip()
    texto = re.sub(r"\s+", " ", texto)

    if texto == "":
        return None

    return texto


def extrair_ano(nome_arquivo):
    """Extrai o ano do nome do arquivo."""
    match = re.search(r"20\d{2}", nome_arquivo)
    return int(match.group()) if match else None


def encontrar_linha_cabecalho(df_bruto):
    """
    Procura automaticamente a linha que contém os meses JAN, FEV, MAR etc.
    Isso evita depender de uma linha fixa de cabeçalho.
    """
    for indice, linha in df_bruto.iterrows():
        valores_linha = [limpar_texto(valor) for valor in linha.values]
        valores_linha = [valor for valor in valores_linha if valor is not None]

        qtd_meses = sum(valor in meses for valor in valores_linha)

        if qtd_meses >= 3:
            return indice

    return None

## 5. Estruturando categoria, subcategoria e indicador

As planilhas possuem linhas que funcionam como títulos de seção, como `SAÚDE`, `EDUCAÇÃO` e `PROFISSIONALIZAÇÃO`. Também existem subgrupos em Educação, como `Ensino regular`, `Ensino EJA` e `SCFV`. Esta função preserva essa hierarquia em colunas separadas.

In [8]:
def adicionar_categoria_subcategoria(df):
    """
    Lê a coluna indicador na ordem original da planilha e cria as colunas:
    - categoria
    - subcategoria
    
    Também remove as linhas que são apenas títulos de categoria ou subcategoria.
    """
    categoria_atual = None
    subcategoria_atual = None

    categorias = []
    subcategorias = []
    remover_linha = []

    for indicador in df["indicador"]:
        indicador_limpo = limpar_texto(indicador)
        indicador_upper = indicador_limpo.upper() if indicador_limpo is not None else None

        # Linhas que representam categoria principal
        if indicador_upper in categorias_lem:
            categoria_atual = indicador_upper
            subcategoria_atual = None
            categorias.append(categoria_atual)
            subcategorias.append(subcategoria_atual)
            remover_linha.append(True)
            continue

        # Linhas que representam subcategoria, principalmente dentro de EDUCAÇÃO
        if indicador_limpo in subcategorias_gerais:
            subcategoria_atual = indicador_limpo
            categorias.append(categoria_atual)
            subcategorias.append(subcategoria_atual)
            remover_linha.append(True)
            continue

        categorias.append(categoria_atual)
        subcategorias.append(subcategoria_atual)
        remover_linha.append(False)

    df = df.copy()
    df["categoria"] = categorias
    df["subcategoria"] = subcategorias
    df["remover_linha"] = remover_linha

    # Remove linhas de título e mantém apenas indicadores mensuráveis
    df = df[df["remover_linha"] == False].drop(columns=["remover_linha"])

    return df

## 6. Leitura e Transformação dos LEM

Esta função transforma cada planilha do formato largo para o formato longo:

De:

`indicador | JAN | FEV | MAR | ... | DEZ`

Para:

`ano | mes_numero | mes | data | categoria | subcategoria | indicador | valor`

In [9]:
def ler_lem_mensal(arquivo):
    """Lê e transforma uma planilha LEM mensal/anual tradicional."""
    ano = extrair_ano(arquivo.name)

    if ano is None:
        raise ValueError(f"Não foi possível identificar o ano no arquivo: {arquivo.name}")

    excel = pd.ExcelFile(arquivo)

    # Procura a primeira aba não vazia que contenha uma linha com os meses
    df_bruto = None
    linha_cabecalho = None
    aba_usada = None

    for aba in excel.sheet_names:
        df_teste = pd.read_excel(arquivo, sheet_name=aba, header=None)

        if df_teste.dropna(how="all").empty:
            continue

        linha_encontrada = encontrar_linha_cabecalho(df_teste)

        if linha_encontrada is not None:
            df_bruto = df_teste
            linha_cabecalho = linha_encontrada
            aba_usada = aba
            break

    if df_bruto is None or linha_cabecalho is None:
        raise ValueError(f"Não foi encontrada uma aba válida com meses no arquivo: {arquivo.name}")

    # Define cabeçalho real
    novo_cabecalho = df_bruto.iloc[linha_cabecalho].apply(limpar_texto).tolist()

    # Mantém os dados abaixo do cabeçalho
    df = df_bruto.iloc[linha_cabecalho + 1:].copy()
    df.columns = novo_cabecalho

    # Remove colunas sem nome
    df = df.loc[:, [col is not None for col in df.columns]]

    # Renomeia a primeira coluna como indicador
    df = df.rename(columns={df.columns[0]: "indicador"})

    # Padroniza nomes das colunas
    df.columns = [limpar_texto(col) for col in df.columns]

    # Mantém apenas indicador + meses existentes
    colunas_meses_existentes = [mes for mes in meses if mes in df.columns]

    if len(colunas_meses_existentes) == 0:
        raise ValueError(f"Nenhuma coluna de mês foi encontrada no arquivo: {arquivo.name}")

    df = df[["indicador"] + colunas_meses_existentes]

    # Remove linhas vazias
    df = df.dropna(how="all")
    df = df.dropna(subset=["indicador"])

    # Limpa texto dos indicadores
    df["indicador"] = df["indicador"].apply(limpar_texto)

    # Adiciona categoria e subcategoria
    df = adicionar_categoria_subcategoria(df)

    # Transforma de formato largo para longo
    df_longo = df.melt(
        id_vars=["categoria", "subcategoria", "indicador"],
        value_vars=colunas_meses_existentes,
        var_name="mes",
        value_name="valor"
    )

    # Converte valores para número
    df_longo["valor"] = pd.to_numeric(df_longo["valor"], errors="coerce")

    # Remove valores vazios ou não numéricos
    df_longo = df_longo.dropna(subset=["valor"])

    # Adiciona colunas temporais
    df_longo["ano"] = ano
    df_longo["mes_numero"] = df_longo["mes"].map(mapa_meses)
    df_longo["data"] = pd.to_datetime(
        df_longo["ano"].astype(str) + "-" + df_longo["mes_numero"].astype(str).str.zfill(2) + "-01"
    )

    # Mantém temporariamente para auditoria; será removida no dataset final
    df_longo["fonte_arquivo"] = arquivo.name
    df_longo["aba_usada"] = aba_usada

    # Organiza colunas
    df_longo = df_longo[[
        "ano",
        "mes_numero",
        "mes",
        "data",
        "categoria",
        "subcategoria",
        "indicador",
        "valor",
        "fonte_arquivo",
        "aba_usada",
    ]]

    return df_longo

## 7. Consolidação em uma Base Única

In [11]:
bases_lem = []
erros_processamento = []

for arquivo in arquivos_excel:
    try:
        df_tratado = ler_lem_mensal(arquivo)
        bases_lem.append(df_tratado)
        print(f"Arquivo processado com sucesso: {arquivo.name} | {df_tratado.shape[0]} registros")
    except Exception as erro:
        erros_processamento.append({"arquivo": arquivo.name, "erro": str(erro)})
        print(f"Erro ao processar {arquivo.name}: {erro}")

if len(bases_lem) == 0:
    raise ValueError("Nenhum arquivo LEM foi processado com sucesso.")

lem_consolidado = pd.concat(bases_lem, ignore_index=True)

lem_consolidado.head(20)

Arquivo processado com sucesso: LEM 2021.xlsx | 389 registros
Arquivo processado com sucesso: LEM 2024.xlsx | 393 registros
Arquivo processado com sucesso: LEM 2025.xlsx | 388 registros
Arquivo processado com sucesso: LEM 2022.xlsx | 388 registros
Arquivo processado com sucesso: LEM 2023.xlsx | 304 registros


,ano,mes_numero,mes,data,categoria,subcategoria,indicador,valor,fonte_arquivo,aba_usada
0,2021,1,JAN,2021-01-01,NaN,NaN,Atendimentos indvidual,40.0,LEM 2021.xlsx,Planilha1
1,2021,1,JAN,2021-01-01,NaN,NaN,Atedimento familiar,31.0,LEM 2021.xlsx,Planilha1
2,2021,1,JAN,2021-01-01,NaN,NaN,Interface com rede socioassistencial,10.0,LEM 2021.xlsx,Planilha1
3,2021,1,JAN,2021-01-01,NaN,NaN,Interface com judiciário,8.0,LEM 2021.xlsx,Planilha1
4,2021,1,JAN,2021-01-01,NaN,NaN,Interface com saúde,18.0,LEM 2021.xlsx,Planilha1
5,2021,1,JAN,2021-01-01,NaN,NaN,Interface com educação,9.0,LEM 2021.xlsx,Planilha1
6,2021,1,JAN,2021-01-01,NaN,NaN,PIAS / Relatórios,17.0,LEM 2021.xlsx,Planilha1
7,2021,1,JAN,2021-01-01,NaN,NaN,Visitas Domiciliares,2.0,LEM 2021.xlsx,Planilha1
8,2021,1,JAN,2021-01-01,NaN,NaN,Apadrinhamento afetivo,0.0,LEM 2021.xlsx,Planilha1
9,2021,1,JAN,2021-01-01,NaN,NaN,Colocação em família substituta,0.0,LEM 2021.xlsx,Planilha1


## 8. Verificação

Verifica se tem apenas uma planilha para cada ano.

In [12]:
versoes_por_ano = (
    lem_consolidado
    .groupby("ano")["fonte_arquivo"]
    .nunique()
    .reset_index(name="qtd_arquivos")
)

versoes_por_ano

,ano,qtd_arquivos
0,2021,1
1,2022,1
2,2023,1
3,2024,1
4,2025,1


## 9. Padronização dos Nomes dos Indicadores

In [14]:
mapa_indicadores = {
    "Atedimento familiar": "Atendimento familiar",
    "Atendimentos indvidual": "Atendimentos individual",
    "Saude mental": "Saúde mental",
    "Saúde Clinica": "Saúde clínica",
    "Saúde Clínica": "Saúde clínica",
    "ENCAMINHADO para mercado de trabalho": "Encaminhados para mercado de trabalho",
    "ENCAMINHADOS para curso profissionalizante": "Encaminhados para curso profissionalizante",
    "INSERIDO no mercado de trabalho": "Inseridos no mercado de trabalho",
    "INSERIDOS em curso profissionalizante": "Inseridos em curso profissionalizante",
    "Pias / Relatórios": "PIAs / Relatórios",
    "PIAS / Relatórios": "PIAs / Relatórios",
}

lem_consolidado["indicador"] = lem_consolidado["indicador"].replace(mapa_indicadores)

# Padroniza categoria e subcategoria
lem_consolidado["categoria"] = lem_consolidado["categoria"].replace({
    "SAUDE": "SAÚDE",
    "Saúde": "SAÚDE",
    "Educação": "EDUCAÇÃO",
    "Profissionalização": "PROFISSIONALIZAÇÃO",
})

lem_consolidado["subcategoria"] = lem_consolidado["subcategoria"].replace({
    "Ensino eja": "Ensino EJA",
    "Scfv": "SCFV",
})

sorted(lem_consolidado["indicador"].dropna().unique())

['Apadrinhamento afetivo',
 'Atendimento familiar',
 'Atendimentos individual',
 'Colocação em família substituta',
 'Desligamentos',
 'Documentação civil (RG / CPF/ CTPS / Certidão de nascimento)',
 'Efetivos na casa',
 'Encaminhados para curso profissionalizante',
 'Encaminhados para mercado de trabalho',
 'Evasão',
 'Inseridos em curso profissionalizante',
 'Inseridos no mercado de trabalho',
 'Interface com educação',
 'Interface com judiciário',
 'Interface com rede socioassistencial',
 'Interface com saúde',
 'Internações',
 'Novos ingressos',
 'Outros: Reforço escolar, psicopedagoga, trabalho educativo',
 'PIAs / Relatórios',
 'Reunião de equipe (técnica, casa, coordenação, gerente)',
 'Saúde clínica',
 'Saúde mental',
 'Transferências',
 'Visitas Domiciliares',
 'aguardando vaga',
 'matriculados']

## 10. Criação de indicador completo

A coluna `indicador_completo` evita confusão em casos como Educação, onde o indicador `matriculados` pode pertencer a subcategorias diferentes, como `Ensino regular`, `Ensino EJA` ou `SCFV`.

In [15]:
def montar_indicador_completo(linha):
    if pd.notna(linha["subcategoria"]):
        return f"{linha['subcategoria']} - {linha['indicador']}"
    return linha["indicador"]

lem_consolidado["indicador_completo"] = lem_consolidado.apply(montar_indicador_completo, axis=1)

lem_consolidado.head(20)

,ano,mes_numero,mes,data,categoria,subcategoria,indicador,valor,fonte_arquivo,aba_usada,indicador_completo
0,2021,1,JAN,2021-01-01,NaN,NaN,Atendimentos individual,40.0,LEM 2021.xlsx,Planilha1,Atendimentos individual
1,2021,1,JAN,2021-01-01,NaN,NaN,Atendimento familiar,31.0,LEM 2021.xlsx,Planilha1,Atendimento familiar
2,2021,1,JAN,2021-01-01,NaN,NaN,Interface com rede socioassistencial,10.0,LEM 2021.xlsx,Planilha1,Interface com rede socioassistencial
3,2021,1,JAN,2021-01-01,NaN,NaN,Interface com judiciário,8.0,LEM 2021.xlsx,Planilha1,Interface com judiciário
4,2021,1,JAN,2021-01-01,NaN,NaN,Interface com saúde,18.0,LEM 2021.xlsx,Planilha1,Interface com saúde
5,2021,1,JAN,2021-01-01,NaN,NaN,Interface com educação,9.0,LEM 2021.xlsx,Planilha1,Interface com educação
6,2021,1,JAN,2021-01-01,NaN,NaN,PIAs / Relatórios,17.0,LEM 2021.xlsx,Planilha1,PIAs / Relatórios
7,2021,1,JAN,2021-01-01,NaN,NaN,Visitas Domiciliares,2.0,LEM 2021.xlsx,Planilha1,Visitas Domiciliares
8,2021,1,JAN,2021-01-01,NaN,NaN,Apadrinhamento afetivo,0.0,LEM 2021.xlsx,Planilha1,Apadrinhamento afetivo
9,2021,1,JAN,2021-01-01,NaN,NaN,Colocação em família substituta,0.0,LEM 2021.xlsx,Planilha1,Colocação em família substituta


## 11. Validações de qualidade da base consolidada

In [17]:
print("Dimensão da base consolidada:")
print(lem_consolidado.shape)

print("Quantidade de registros por ano:")
display(lem_consolidado["ano"].value_counts().sort_index())

print("Quantidade de registros por categoria:")
display(lem_consolidado["categoria"].value_counts(dropna=False))

print("Quantidade de indicadores únicos:")
print(lem_consolidado["indicador_completo"].nunique())

Dimensão da base consolidada:
(1862, 11)
Quantidade de registros por ano:


ano
2021    389
2022    388
2023    304
2024    393
2025    388
Name: count, dtype: int64

Quantidade de registros por categoria:


categoria
NaN                   1012
EDUCAÇÃO               436
PROFISSIONALIZAÇÃO     240
SAÚDE                  174
Name: count, dtype: int64

Quantidade de indicadores únicos:
33


In [18]:
# Valores ausentes por coluna
nulos = (
    lem_consolidado
    .isna()
    .sum()
    .reset_index()
)
nulos.columns = ["coluna", "qtd_nulos"]
nulos["percentual_nulos"] = (nulos["qtd_nulos"] / len(lem_consolidado) * 100).round(2)

nulos

,coluna,qtd_nulos,percentual_nulos
0,ano,0,0.00
1,mes_numero,0,0.00
2,mes,0,0.00
3,data,0,0.00
4,categoria,1012,54.35
5,subcategoria,1426,76.58
6,indicador,0,0.00
7,valor,0,0.00
8,fonte_arquivo,0,0.00
9,aba_usada,0,0.00


In [20]:
# Verificação de duplicatas na chave lógica dos dados
chave = ["ano", "mes_numero", "categoria", "subcategoria", "indicador"]

duplicatas = lem_consolidado[lem_consolidado.duplicated(subset=chave, keep=False)]

print(f"Quantidade de registros duplicados pela chave lógica: {duplicatas.shape[0]}")

Quantidade de registros duplicados pela chave lógica: 0


## 12. Remoção de colunas auxiliares e organização final

As colunas `fonte_arquivo` e `aba_usada` foram úteis para auditoria durante o pré-processamento, mas não serão mantidas no dataset final.

In [21]:
colunas_finais = [
    "ano",
    "mes_numero",
    "mes",
    "data",
    "categoria",
    "subcategoria",
    "indicador",
    "indicador_completo",
    "valor",
]

lem_final = lem_consolidado[colunas_finais].copy()

# Ordena a base final
lem_final = lem_final.sort_values([
    "ano",
    "mes_numero",
    "categoria",
    "subcategoria",
    "indicador",
]).reset_index(drop=True)

lem_final.head(30)

,ano,mes_numero,mes,data,categoria,subcategoria,indicador,indicador_completo,valor
0,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino EJA,aguardando vaga,Ensino EJA - aguardando vaga,0.0
1,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino EJA,matriculados,Ensino EJA - matriculados,0.0
2,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino infantil,aguardando vaga,Ensino infantil - aguardando vaga,3.0
3,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino infantil,matriculados,Ensino infantil - matriculados,2.0
4,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino regular,aguardando vaga,Ensino regular - aguardando vaga,3.0
5,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino regular,matriculados,Ensino regular - matriculados,12.0
6,2021,1,JAN,2021-01-01,EDUCAÇÃO,SCFV,"Outros: Reforço escolar, psicopedagoga, trabal...","SCFV - Outros: Reforço escolar, psicopedagoga,...",0.0
7,2021,1,JAN,2021-01-01,EDUCAÇÃO,SCFV,aguardando vaga,SCFV - aguardando vaga,0.0
8,2021,1,JAN,2021-01-01,EDUCAÇÃO,SCFV,matriculados,SCFV - matriculados,0.0
9,2021,1,JAN,2021-01-01,PROFISSIONALIZAÇÃO,NaN,Encaminhados para curso profissionalizante,Encaminhados para curso profissionalizante,0.0


## 13. Exportação do dataset final

In [22]:
pasta_saida = raiz_projeto / "dados_tratados"
pasta_saida.mkdir(exist_ok=True)

caminho_csv = pasta_saida / "lem_consolidado_2021_2025.csv"
caminho_excel = pasta_saida / "lem_consolidado_2021_2025.xlsx"

lem_final.to_csv(caminho_csv, index=False, encoding="utf-8-sig")
lem_final.to_excel(caminho_excel, index=False)

print("Dataset consolidado salvo com sucesso!")
print(f"CSV: {caminho_csv}")
print(f"Excel: {caminho_excel}")

Dataset consolidado salvo com sucesso!
CSV: /Users/barbaradapper/Library/Mobile Documents/com~apple~CloudDocs/Desktop/2026/Visualização/TE-Visualizacao/dados_tratados/lem_consolidado_2021_2025.csv
Excel: /Users/barbaradapper/Library/Mobile Documents/com~apple~CloudDocs/Desktop/2026/Visualização/TE-Visualizacao/dados_tratados/lem_consolidado_2021_2025.xlsx


## 14. Conferência

In [23]:
print("Resumo final da base tratada")
print("- Linhas:", lem_final.shape[0])
print("- Colunas:", lem_final.shape[1])
print("- Anos:", sorted(lem_final["ano"].unique()))
print("- Categorias:", sorted(lem_final["categoria"].dropna().unique()))
print("- Indicadores completos:", lem_final["indicador_completo"].nunique())

lem_final.sample(min(10, len(lem_final)), random_state=42)

Resumo final da base tratada
- Linhas: 1862
- Colunas: 9
- Anos: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
- Categorias: ['EDUCAÇÃO', 'PROFISSIONALIZAÇÃO', 'SAÚDE']
- Indicadores completos: 33


,ano,mes_numero,mes,data,categoria,subcategoria,indicador,indicador_completo,valor
233,2021,8,AGO,2021-08-01,EDUCAÇÃO,Ensino infantil,matriculados,Ensino infantil - matriculados,7.0
530,2022,5,MAI,2022-05-01,PROFISSIONALIZAÇÃO,NaN,Encaminhados para curso profissionalizante,Encaminhados para curso profissionalizante,0.0
1220,2024,5,MAI,2024-05-01,EDUCAÇÃO,SCFV,matriculados,SCFV - matriculados,0.0
471,2022,3,MAR,2022-03-01,NaN,NaN,Apadrinhamento afetivo,Apadrinhamento afetivo,12.0
415,2022,1,JAN,2022-01-01,NaN,NaN,Interface com rede socioassistencial,Interface com rede socioassistencial,10.0
631,2022,8,AGO,2022-08-01,PROFISSIONALIZAÇÃO,NaN,Inseridos no mercado de trabalho,Inseridos no mercado de trabalho,0.0
1441,2024,11,NOV,2024-11-01,NaN,NaN,Transferências,Transferências,0.0
414,2022,1,JAN,2022-01-01,NaN,NaN,Interface com judiciário,Interface com judiciário,8.0
887,2023,5,MAI,2023-05-01,NaN,NaN,Colocação em família substituta,Colocação em família substituta,2.0
1576,2025,4,ABR,2025-04-01,EDUCAÇÃO,Ensino infantil,matriculados,Ensino infantil - matriculados,0.0
